# Numerical Integration of Geodesics in the Kerr Metric

Manuela Marín Gómez — Universidad de Antioquia, 2025.

**Portfolio edition:** the original individual solution is retained; assignment instructions and personal identification details are omitted. Some models and figures require review (see `REVIEW_NOTES.md`).


In [ ]:
!pip install -Uq einsteinpy pymcel

In [ ]:
import sympy as sp
import numpy as np
import pymcel as pc
import matplotlib.pyplot as plt
import einsteinpy.symbolic as es
from scipy.integrate import solve_ivp
from mpl_toolkits.mplot3d import Axes3D

# **1). Aproximación a la métrica de Schwarzschild**

La métrica de Kerr está dada por:

$$d s^2= \left(1-\frac{2 s r}{\Sigma}\right) c^2 d t^2+\frac{4 a c s^2 r \sin ^2 \theta}{\Sigma} d t d \phi-\frac{\Sigma}{\Delta} d r^2-\Sigma d \theta^2 -\frac{\sin ^2 \theta}{\Sigma}\left[\left(r^2+(a s)^2\right)^2-(a s)^2 \Delta \sin ^2 \theta\right] d \phi^2 $$

donde
 $$\Sigma=r^2+(a s)^2 \cos ^2 \theta, \quad \Delta=r^2-2 s r+(a s)^2 $$

Vemos que la métrica de Kerr describe un cuerpo en rotación. El momento angular total del cuerpo es $J$, y el parámetro adimensional de Spin es $a = c J/(G M^2)$, de este modo, si el momento angular es cero, entonces $J = 0$, lo que implica que el parámetro de spin es $a = 0$.

Si sustituimos entonces el valor del parámetro del spin en los parámetros auxiliares tenemos que:

$$
Σ = r^2, \quad Δ = r^2 -2sr
$$

Analicemos entonces las componentes de la métrica de Kerr cuando el momento angular es cero.


## Componente temporal $g_{tt}$

Tomando la forma general:

$$g_{tt} = \left(1-\frac{2 s r}{\Sigma}\right)$$

si $Σ=r^2$ entonces:

$$g_{tt} = \left( 1- \frac{2sr}{r^2}\right)$$

Simplicando el factor se teiene que:

$$g_{tt} = \left( 1- \frac{2s}{r}\right)$$


pero sabemos que el parámetro de Schwarzschild $s$ está dado por:

$$s = \frac{GM}{c^2}$$

De este modo la componente temporal será:

$$g_{tt} = \left( 1-\frac{2GM}{c^2r} \right)$$

vemos que si la rotación es cero, la dependencia angular desaparece y el tiempo sólo sependerá del radio

## Componente radial $g_{rr}$


El término radial viene dado por

$$ g_{rr} = -\frac{Σ}{Δ}$$

remplazando los valores en el límite cuando $a=0$ se tiene que:

$$
g_{rr} = - \frac{r^2}{r^2-2sr}
$$

factorizando

$$
g_{rr} = - \frac{1}{1-\frac{2s}{r}}
$$

Sustituyendo nuevamente el valor del parámetro de Schwarzschild:

$$
g_{rr} = -\left( 1 -\frac{2GM}{c^2}\right)^{-1}
$$

## Componente angular polar $g_{θ θ}$

El término asociado a la componente angular polar es:

$$g_{θθ} = - Σ$$

En el límite cuando $a=0$ se obtenine:

$$
g_{θθ} = - r^2
$$

## Componente azimutal $g_{\phi \phi}$

La componente azimutal está dada por:

$$g_{\phi\phi} = -\frac{\sin^2\theta}{\Sigma} \left[ \left(r^2+(as)^2\right)^2 - (as)^2 \Delta \sin^2\theta \right]$$

Si $a=0$, entonces $(as)^2=0$, por tanto se anulan varios términos, luego:


$$g_{\phi\phi} = -\frac{\sin^2\theta}{r^2} r^4$$

finalmente

$$g_{\phi\phi}= -r^2\sin^2$$

##  Componente $g_{t\phi}$

Finalmente podemos ver que cuanod la componente angular es nular:

$$\frac{4 a c s^2 r \sin ^2 \theta}{\Sigma} = 0$$

es decir , el término cruzado se anula, lo que significa la eliminación del acoplamiento espacio-tiempo característico de la rotación.

 Veamos que podemos escribir la métrica de Kerr como la suma explícita:$$d s^2 = g_{tt} (c dt)^2 + g_{rr} d r^2 + g_{\theta \theta} d \theta^2 + g_{\phi \phi} d \phi^2 + 2 g_{t \phi} (c dt) d \phi$$

Sabemos además que la métrica de Schwarzschild describe el espacio-tiempo curvo en el exterior de un cuerpo masivo, estático y esféricamente simétrico que no rota.
En coordenadas esféricas $(t, r, \theta, \phi)$, la métrica de Schwarzschild está dada por:

$$d s^2 = \left(1-\frac{2 G M}{c^2 r}\right) c^2 d t^2 - \left( 1 - \frac{2 G M}{c^2 r} \right)^{-1} d r^2 - r^2 \left(d \theta^2 + \sin ^2 \theta d \phi^2\right)$$

De este modo, remplazando las componente en el límite cuando $a=0$ en la métrica de Kerr obtenemos:



$$d s^2 = \left(1-\frac{2 s}{r}\right) c^2 d t^2 + (0) d t d \phi - \left( 1 - \frac{2 s}{r} \right)^{-1} d r^2 - r^2 d \theta^2 - r^2 \sin ^2 \theta d \phi^2
$$Reorganizando y usando la definición del parámetro de Schwarzschild $s = GM/c^2$:
$$d s^2 = \left(1-\frac{2 G M}{c^2 r}\right) c^2 d t^2 - \left( 1 - \frac{2 G M}{c^2 r} \right)^{-1} d r^2 - r^2 \left(d \theta^2 + \sin ^2 \theta d \phi^2\right)$$

Podemos ver que esta es exactamente la métrica de Schwarzschild, que describe el espaciotiempo alrededor de una masa $M$ no rotante, es decir que  la métrica de Kerr es la generalización de la métrica de Schwarzschild para objetos en rotación.

##**Desarrollo usando `einsteinpy`**

Verifiquemos esto usando `einsteinpy`

In [ ]:
# Definimos las variables simbólicas
t, r, theta, phi = sp.symbols("t r theta phi")
# Parámetros físicos
s, a, c = sp.symbols("s a c")

In [ ]:
# Definimos las variables auxiliares
Sigma = r**2 + (a * s)**2 * sp.cos(theta)**2
Delta = r**2 - 2 * s * r + (a * s)**2

In [ ]:
# Construimos la matriz del tensor métrico
# Componente dt^2
g_tt = (1 - (2 * s * r) / Sigma) * c**2

# Componente dr^2
g_rr = -Sigma / Delta

# Componente dtheta^2
g_thetatheta = -Sigma

# Componente dphi^2
term_phi = ((r**2 + (a * s)**2)**2 - (a * s)**2 * Delta * sp.sin(theta)**2)
g_phiphi = -(sp.sin(theta)**2 / Sigma) * term_phi

# Componentes cruzados dt dphi (y dphi dt por simetría)
g_tphi = (2 * a * c * s**2 * r * sp.sin(theta)**2) / Sigma
g_phit = g_tphi

In [ ]:
# Creamos la lista de listas (matriz 4x4)
# Asumiendo que los términos fuera de la diagonal son 0
matriz_metrica1 = [
    [g_tt, 0, 0, g_tphi],
    [0, g_rr, 0, 0],
    [0, 0, g_thetatheta, 0],
    [g_phit, 0, 0, g_phiphi]
]

metrica_kerr1 = es.MetricTensor(matriz_metrica1, (t, r, theta, phi))

In [ ]:
metrica_kerr1.tensor()

Apliquemos entonces el límite cuando el momento angular $J=0$,  es decir  cuando $a=0$.

In [ ]:
schwarzschild = metrica_kerr1.tensor().subs(a, 0) # Se remplaza 0 donde está a
sp.simplify(schwarzschild )

Podemos observar entonces que cuando $a=0$, la métrica de Kerr tiende a la métrica de Schwarzschild

# **2). Métrica de Kerr linealizada y ecuaciones de campo**

Supongamos que el parámetro adimensional del spin satisface:

$$|a| \ll 1$$

Trabajaremos entonces solo hasta el primer orden en $a$ y se despreciarán los términos $\mathcal{O}(a^2)$, de este modo podemos simplicar las funciones auxiliares tal que:

$$\Sigma = r^2 + a^2 \cos^2\theta \approx r^2 + \mathcal{O}(a^2) $$

es decir $$Σ ≈ r^2$$

De forma similar:

$$\Delta = r^2 - 2 M r + a^2 \approx r^2 - 2 M r + \mathcal{O}(a^2)$$

por tanto

$$Δ ≈ r^2-2sr$$

Analicemos entonces los componentes de la métrica.

## Componente $g_{tt}$

$$g_{tt} = \left(1 - \frac{2 s r}{\Sigma}\right) c^2 ≈ \left(1 - \frac{2 s r}{r^2}\right)c^2 = \left(1 - \frac{2s}{r}\right)c^2 $$


Podemos ver que este es exactamente el componente temporal de la métrica de Schwarzschild.

## Término cruzado $g_{t\phi}$

En la forma del intervalo, el término cruzado es

$$\frac{4 a c s^2 r \sin^2\theta}{\Sigma} dt\, d\phi$$
Pero recordemos que el elemento de línea de forma general está dado por:

$$
ds^2 =g_{tt}dt^2 + 2g_{t\phi}dtd\phi + g_{\phi\phi}d\phi^2 + g_{rr}dr^2 + g_{\theta\theta}d\theta^2
$$

por tanto
$$2 g_{t\phi} = \frac{4 a c s^2 r \sin^2\theta}{\Sigma}$$

entonces

$$g_{t\phi} = \frac{2 a c s^2 r \sin^2\theta}{\Sigma}$$

Luego, a primer orden en $a$se tiene que:
$$g_{t\phi} \simeq \frac{2 a c s^2 r \sin^2\theta}{r^2} = \frac{2 a c s^2 \sin^2\theta}{r} $$




## Componente radial $g_{rr}$

El componente radial de la métrica es:
$$g_{rr} = -\frac{\Sigma}{\Delta}$$

Usando las aproximaciones tenemos que:

$$g_{rr} \approx -\frac{r^2}{r^2 - 2sr} = -\left( 1 -\frac{2s}{r} \right)^{-1}$$

Podemos ver nuevamente que este es exactamente el componente radial de la métrica de Schwarzschild.

## Componentes angulares $g_{\theta\theta}$ y $g_{\phi\phi}$

Para $g_{\theta\theta}$:$$g_{\theta\theta} = -\Sigma \simeq -r^2$$como en la métrica de Schwarzschild.

Para $g_{\phi\phi}$ tenemos:
$$g_{\phi\phi} = -\frac{\sin^2\theta}{\Sigma} \left[ \left(r^2 + (as)^2\right)^2 - (as)^2 \Delta \sin^2\theta \right]$$

A primer orden en $a$ podemos descartar todos los términos que contengan $(as)^2$, porque ya son de orden $\mathcal{O}(a^2)$.

Por tanto:
$$\left(r^2 + (as)^2\right)^2 \simeq r^4$$$$(as)^2 \Delta \sin^2\theta \simeq 0$$$$\Sigma \simeq r^2$$

entonces:
$$g_{\phi\phi} \simeq -\frac{\sin^2\theta}{r^2} \, r^4 = - r^2 \sin^2\theta$$

que coincide con Schwarzschild.


De este modo, la métrica de Kerr para $|a| \ll 1$ será:

$$ds^2 \approx \left(1-\frac{2s}{r}\right)c^2 dt^2 - \left(1-\frac{2s}{r}\right)^{-1} dr^2 - r^2 d\theta^2 - r^2 \sin^2\theta \, d\phi^2 + \frac{4 a c s^2 \sin^2\theta}{r} \, dt \, d\phi $$

Matricialmente podemos ver esto como:

$$g_{\mu\nu} = \begin{pmatrix}
\left(1 - \dfrac{2s}{r}\right)c^2 & 0 & 0 & \dfrac{2 a c s^2}{r}\sin^2\theta \\[10pt]
0 & -\left(1 - \dfrac{2s}{r}\right)^{-1} & 0 & 0 \\[10pt]
0 & 0 & -r^2 & 0 \\[10pt]
\dfrac{2 a c s^2}{r}\sin^2\theta & 0 & 0 & -r^2\sin^2\theta
\end{pmatrix} + \mathcal{O}(a^2)$$

Por facilidad, definiendo $T=ct$, de modo que $dT =cdt$, podemos redefinir la métrica  como:

$$
d s^2= \left(1-\frac{2 s }{r}\right)dT + \frac{(4 a  s^2  \sin ^2 \theta)}{r} dT d \phi-\frac{r}{(r-2s)} d r^2- r^2 d \theta^2 - r^2sin^2\theta d\phi^2
$$

O matricialmeten esto se puede ver como:


$$
g_{\mu\nu} =
\begin{pmatrix}
\left( 1 - \frac{2s}{r} \right) & 0 & 0 & \left( \frac{2as^2sin^2\theta}{r}\right) \\
0 & \left(-\frac{r}{r-2s}\right) & 0 &0 \\
0 & 0 & \left(-r^2\right) & 0 \\
\left( \frac{2as^2sin^2\theta}{r}\right) & 0 & 0 & \left(-r^2sin^2\theta\right)
\end{pmatrix} + \mathcal{O}^2
$$


Veamoss esto usando `einsteinpy` (trabajando por simplicidad con $T=ct$).


In [ ]:
T, r, theta, phi, a, s = sp.symbols("T r theta phi a s")

In [ ]:
# Componnetes de la matriz del tensor métrico en el límite
#Componente temporal
g_TT = 1 - 2 * s / r

# Componente radial
g_RR = - r / (r - 2 * s)

# Componentes angulares
g_THETA = - r**2 #g_thetatheta
g_PHI = - r**2 * sp.sin(theta)**2 #g_phiphi

# Término cruzado
g_TPHI = 2 * a * s**2 * sp.sin(theta)**2 / r

In [ ]:
# Matriz del tensor métrico en el límite lineal en a
g_munu_lin = [
    [g_TT, 0, 0, g_TPHI],
    [0, g_RR, 0, 0],
    [0, 0, g_THETA, 0],
    [g_TPHI, 0, 0, g_PHI]
]

g_munu_metrica_lin = es.MetricTensor(g_munu_lin, (T, r, theta, phi))

In [ ]:
g_munu_metrica_lin.tensor()

## Ecuaciones del campo en el vacío

Sabemos que el campo gravitacional en la relatividad general está descrito por el tensor de Einstein:

$$
G_{\mu\nu} = R_{\mu\nu}- \frac{1}{2}Rg_{\mu\nu}
$$

En el vacío(donde el tensor de energía-momento $T_{\mu\nu} = 0$), las ecuaciones de campo satisfacen:

$$G_{\mu\nu}(g) =0$$

esto implica que el espacio-tiempo está curvado solo por su propia geometría y las condiciones de frontera.

De este modo, como solución exacta, la métrica de Kerr satisface las ecuaciones de vacío para cualquier valor de rotación permitido:

$$G_{\mu\nu}(g_{Kerr}) = 0$$






La métrica de Kerr depende del parámetro de rotación, si la expandimos en series de Taylor alrededor de $a=0$ tenemos que:

$$g_{\mu\nu}(a) = g_{\mu\nu}^{(0)} + a h_{\mu\nu} + a^2 k_{\mu\nu} + ... $$

Pero como queremos trabajar solo hasta el orden lineal $a$, podemos despreciar los términos de orden superior, de modo que:

$$g_{\mu\nu}(a) = g_{\mu\nu}^{(0)} + a h_{\mu\nu} + \mathcal{O}(a^2)$$

veamos que $g_{\mu\nu}^{(0)}$ es la métrica de Schwarzschild y $h_{\mu\nu}$ es la perturbación lineal debido a la rotación.

El tensor de Einstein $G_{\mu\nu}$ es no lineal respecto respecto a la métrica $g$, pero podemos expandirlo alrededor de la solución de vacío de Schwarzschild:

$$G_{\mu\nu}(g_{Kerr}) = G_{\mu\nu}(g^{(0)} + ah + \mathcal{O}(a^2) = G_{\mu\nu}(g^{(0)} + a δG_{\mu\nu}(h) $$

Pero como Schwarzschild es solución del vacío,se cumple que:

$$
G_{\mu\nu}(g^{(0)}) = 0
$$

Por tanto, para que la métrica aproximada (lineal en $a$) siga siendo solución de las ecuaciones de campo en el vacío hasta primer orden en $a$, se debe cumpli necesariamente que:

$$
δG_{\mu\nu}(h) =0
$$

es decir, el término lineal en
$a$ del tensor de Einstein debe anularse (dentro del error de truncamiento).

Verifiquemos que eso se cumpla

In [ ]:
# Tensor de Einstein para la métrica de Kerr en el límite lineal
G = es.EinsteinTensor.from_metric(g_munu_metrica_lin)
G_tensor = G.tensor()
G_tensor

Podemos truncar entonces los términos de orden mayor a 1 para verificar que el tensor de Einstein sea nulo

In [ ]:
# Matriz donde guardaremos G truncado a orden lineal en a
G_lin_order = sp.MutableDenseMatrix([[0]*4 for _ in range(4)])

for i in range(4):
  for j in range(4):
    # Serie de G en a alrededor de a=0 hasta orden a^1
    series_ij = sp.series(G_tensor[i, j], a, 0, 2)  # incluye términos en a^0 y a^1
    # removeO() elimina el término O(a^2)
    G_lin_order[i, j] = sp.simplify(series_ij.removeO())

G_lin_order

Podemos ver que todas las componentes de $G_{\mu\nu}$ hasta orden lineal en $a$ son nulas, por lo cual la métrica aproximada (lineal en $a$) satisface las ecuaciones de campo en el vacío hasta primer orden en el parámetro de rotación.

# **3).Símbolos de Christoffel y ecuaciones de la geodésica para el límite lineal**

## Símbolos de Christoffel

Los símbolos de Christoffel se definen como:$$\Gamma^\mu_{\ \alpha\beta} = \frac{1}{2} g^{\mu\nu} \left( \partial_\alpha g_{\nu\beta} + \partial_\beta g_{\nu\alpha} - \partial_\nu g_{\alpha\beta} \right)$$

Con einsteinpy podemos obtenerlos de forma simbólica a partir de la métrica

In [ ]:
# Símbolos de Christoffel en el límite linela
Gamma = es.ChristoffelSymbols.from_metric(g_munu_metrica_lin)
Gamma_tensor = Gamma.tensor()

In [ ]:
Gamma_tensor

Veamos que esta matriz puede ser simplificada para facilidad en los cálculos

In [ ]:
Gamma_sim = Gamma_tensor.applyfunc(sp.simplify)
Gamma_sim

Aunque $g_{\mu\nu}$ solo tiene términos hasta orden $a$, al invertir la métrica para obtener $g^{\mu\nu}$ aparecen términos en $a^2$, por lo cual debemos expandir nuevamente en series de potencias:

$$\Gamma^\mu_{\ \alpha\beta}(a) = \Gamma^{(0)\mu}_{\ \ \alpha\beta} + a\,\delta\Gamma^\mu_{\ \alpha\beta} + \mathcal{O}(a^2),$$

y nos quedamos solo con los términos hasta $a^1$.

De este modo, los símbolos de Christoffel para lamétrica de Kerr linela en $a$ serán:

In [ ]:
# Almacenamos los símbolos truncados
Gamma_lin = sp.MutableDenseNDimArray.zeros(4, 4, 4)

# Iteramos sobre todos los índices
for mu in range(4):
  for alfa in range(4):
    for beta in range(4):
      # Realizamos expasión en series de Taylor alrededor de a=0
      series_ij = sp.series(Gamma_tensor[mu, alfa, beta], a, 0, 2) # a^0 y a^1
      Gamma_lin[mu, alfa, beta] = sp.simplify(series_ij.removeO()) # Elimina O^2

Gamma_lin

## Cálculo de las geodésicas

Sabemos que las ecuaciones de la geodésica están dadas por:

$$\frac{d^2 x^\mu}{dτ^2} + \Gamma^\mu_{\ \alpha\beta}\frac{dx^\alpha}{dτ}\frac{dx^\beta}{dτ} = 0$$

donde $x^\mu=(T,r,\theta,\phi)$.



El problema es que para integrar numéricamente las ecuaciones de la geodésica, necesitamos pasar las funciones simbólicas a funciones numéricas, para hacer esto podemos usar la función lambdify de SymPy que  toma las expresiones simbólicas (en este caso, los componentes del tensor métrico y los símbolos de Christoffel $\Gamma^\mu_{\alpha\beta}$) y las compila en funciones de Python que operan sobre arrays de NumPy.

In [ ]:
# Función numérica de la métrica de Kerr en el límite
vars_g, g_munu_fun = g_munu_metrica_lin.tensor_lambdify()
print("Orden variables de la métrica", vars_g)

La estrategia de usar tensor_lambdify para pasar de expresiones simbólicas a funciones numéricas fue tomada de la documentación oficial de EinsteinPy sobre Lambdify symbolic calculation:

https://docs.einsteinpy.org/en/stable/examples/Lambdify%20symbolic%20calculation.html

In [ ]:
# Creamos una función que devuelve la métrica evaluada numéricamente
def g_numerico(T_, r_, theta_, phi_, s_, a_):
  # Asociamos cada símbolo a su valor numérico
  subs = {T: T_, r: r_, theta: theta_, phi: phi_, s: s_, a: a_}
  args = [] # Lista para guardar los valores numéricos
  # Construimos la tupla en el orden de las variables
  for var in vars_g:
    valor = subs[var]
    args.append(valor)
  # Desempaqueta la lista)
  g_num = np.array(g_munu_fun(*args), dtype=float)
  return g_num

In [ ]:
# Este código fue creado con ayuda de ChatGPT
# Prompt: "Ayúdame a descomponer el tensor de rango 3 en matrices 2D para permitir
# el uso de tensor_lambdify. Además ensambla cada matriz en una única función.""

# Función numérica para los símbolos de Christoffel
Gamma_fun = []
Gamma_vars = []

for mu in range(4):
  # Se toma el bloque 4x4 con índices (α,β)
  Gamma_mu_matrix = sp.Matrix(Gamma_lin[mu, :, :])
  # Crea un objeto MetricTensor pero lo rellena con los datos de los símbolos de Christoffel.
  metrica_auxiliar = es.MetricTensor(Gamma_mu_matrix.tolist(), (T, r, theta, phi))
  # Función numérica asociada a esa "métrica" y lista de variables
  vars_gamma, gamma_fun = metrica_auxiliar.tensor_lambdify()
  Gamma_fun.append(gamma_fun)
  Gamma_vars.append(vars_gamma)
  print(f"Variables de Gamma_fun[{mu}]:", vars_gamma)

In [ ]:
# Genermos la función numérica para los símbolos de Christoffel en el límite
def Gamma_numerico(T_, r_, theta_, phi_, s_, a_):
  # Asociamos cada objeto simbólico con su valor numérico
  subs = {T: T_, r: r_, theta: theta_, phi: phi_, s: s_, a: a_}
  # Creamos un cubo de ceros de 4x4x4 para llenar capa por capa
  Gamma_num = np.zeros((4, 4, 4))
  # Iteramos sobre las 4 capas del tensor (t, r, θ, φ)
  for mu in range(4):
    args = [] # Lista para acumular los valores numéricos
    for var in Gamma_vars[mu]:
      valor = subs[var]
      args.append(valor)
    Gamma_num[mu, :, :] = np.array(Gamma_fun[mu](*args))
  return Gamma_num

En términos de componentes podemos definir el vector de estado de las ecuaciones de la geodésica como:

$$Y=(T,\, r,\, \theta,\, \phi,\, U^T,\, U^r,\, U^\theta,\, U^\phi)$$

donde $U^\mu=dx^\mu/dτ$ es la cuadrivelocidad.

De este modo, el sistema de EDOs de primer orden se puede dividir en una parte cinemática y una parte dinámica:

**Cinemática:**

$$\frac{dT}{dτ} = U^T,\quad \frac{dr}{dτ} = U^r,\quad \frac{d\theta}{dτ} = U^\theta,\quad \frac{d\phi}{dτ} = U^\phi$$

**Dinámica (geodésica):**

La aceleración de la partícula está determinada por la geometría del espacio-tiempo mediante:

$$\frac{dU^\mu}{d\tau} = -\Gamma^\mu_{\ \alpha\beta} U^\alpha U^\beta$$

Debido a la convención de suma de Einstein, cada componente de la aceleración involucra una doble suma sobre los índices $\alpha, \beta \in \{0, 1, 2, 3\}$:

$$\frac{dU^\mu}{d\tau} = -\sum_{\alpha=0}^3 \sum_{\beta=0}^3 \Gamma^\mu_{\ \alpha\beta} U^\alpha U^\beta$$

Teniendo esto en cuenta y con las funciones numéricas podemos definir entonces el sistema de ecuaciones diferenciales de la geodésica:

In [ ]:
def sistema_geodesicas(lamda, Y, s_, a_):
  # Definimos las coord. de posición y velocidad
  T_, r_, theta_, phi_, UT, UR, Utheta, Uphi = Y
  U = np.array([UT, UR, Utheta, Uphi])

  # Calculamos los símbolos de Christoffel usando la función numérica
  Gamma = Gamma_numerico(T_, r_, theta_, phi_, s_, a_)

  # Calculamos la cuadriaceleración de la partícula
  dU = np.zeros(4)
  # Suma de Einstein (doble sumatoria)
  for mu in range(4):
    suma = 0
    for alpha in range(4):
      for beta in range(4):
        suma += Gamma[mu, alpha, beta] * U[alpha] * U[beta]
    dU[mu] = -suma

  dYdtau = np.zeros(8)
  dYdtau[:4] = U # Componente cinemática
  dYdtau[4:] = dU # Componente dinámica
  return dYdtau

### Condiciones iniciales en una órbita ecuatorial

**Condiciones iniciales**

En coordenadas esféricas (y en Boyer–Lindquist para la parte angular), las coordenadas cartesianas se relacionan con $(r,\theta,\phi)$ mediante:

$$x = r\sin\theta\cos\phi, \quad y = r\sin\theta\sin\phi, \quad z = r\cos\theta
$$

Sabemos que en el plano ecuatorial se cumple $z=0$, por tanto  $z=r\cos\theta=0$ y si $r\neq 0$, entonces:

$$\cos\theta = 0$$

o equivalentemente
$$
\theta = \frac{\pi}{2}
$$

Por lo tanto, si la partícula sigue una órbita ecualtorial entonces a lo largo de toda la trayectoria siempre se cumple que:

$$\theta(\tau) = \frac{\pi}{2}$$

y en particular, en el instante inicial:

$$\theta_0 = \theta(0) = \frac{\pi}{2}$$

Sabemos además que la métrica de Kerr es axisimétrica, lo que implica que sus componentes no dependen explícitamente de la coordenada $\phi$ ($\partial_\phi g_{\mu\nu} = 0$), esto significa que el sistema es invariante ante rotaciones alrededor del eje de simetría.

Por lo tanto, físicamente, la elección del origen del ángulo azimutal es arbitraria, por tanto, por simplicidad definimos la dirección radial donde comienza la órbita como:

$$\phi_0 = 0$$

Por otro lado, la métrica de Kerr es también estacionaria, lo que significa que no depende explícitamente del tiempo coordenado $T$ ($\partial_T g_{\mu\nu} = 0$), por tanto definiremos el origen del tiempo coordeado como:

$$T_0 = 0$$

**Frecuencia angular de una órbita ecuatorial circular**

Queremos que, en el instante inicial, la órbita sea casi circular en $r=r_0$ y permanezca en el plano ecuatorial, de moso que imponemos que inicialmente no haya componentes radial ni polar de la cuadrivelocidad:

$$U^r_0 = \frac{dr}{d\tau}\bigg|_{\tau=0} = 0, \qquad U^\theta_0 = \frac{d\theta}{d\tau}\bigg|_{\tau=0} = 0
$$

Sabemos además que la ecuación de la geodésica es:

$$\frac{d^2 x^\mu}{d\tau^2} + \Gamma^\mu_{\alpha\beta} \frac{dx^\alpha}{d\tau} \frac{dx^\beta}{d\tau} = 0 $$

Para la componente radial $\mu = r$ , en una órbita circular estacionario se tiene que cumplir que $\ddot{r} = 0$ y  $\dot r = \dot\theta = 0$, de este modo la ecuación radial se reduce a:

$$ \Gamma^r_{\alpha\beta} \frac{dx^\alpha}{d\tau} \frac{dx^\beta}{d\tau} = 0$$

En la órbita ecuatorial solo son no nulas las derivadas respecto a $\tau$ de $T$ y $\phi$,por tanto, en la suma $\Gamma^r_{\alpha\beta}\dot{x}^\alpha \dot{x}^\beta$ los únicos términos que sobreviven son aquellos donde ambos índices $\alpha$ y $\beta$ orresponden a componentes con derivada no nula, es decir $T$ o $\phi$, por tanto, todos los términos con $\alpha$ o $\beta$ igual a $r$ o $\theta$ se anulan, es decir que obtenemos:

$$0 = \Gamma^r_{TT}\dot T^2 + 2\Gamma^r_{T\phi}\dot T\dot\phi + \Gamma^r_{\phi\phi}\dot\phi^2 \tag{1}$$

De este modo podemos definir entonces la frecuencia angular con respecto al tiempo coordenado como :

$$\Omega = \frac{d\phi}{dT} = \frac{\dot{\phi}}{\dot{T}} $$

Dividiendo  la ecuación ($1$) entre $\dot T^2$ (suponienod que $\dot T \neq 0$) se tiene:

$$
\Gamma^r_{T} + 2\Gamma^r_{T\phi}\Omega + \Gamma^r_{\phi\phi}\Omega^2 = 0 \tag{2}
$$


Vemos que esta es una ecuación cuadrática para $\Omega$.

**Expresión de $Γ_{TT}^r$, $Γ_{T\phi}^r$, $Γ_{\phi \phi}^r$**

Sabemos que los símbolos de Christoffel están dados por:

$$
\Gamma^\mu_{\alpha\beta} = \frac{1}{2}g^{\mu\nu} \big(\partial_\alpha g_{\nu\beta} + \partial_\beta g_{\nu\alpha} - \partial_\nu g_{\alpha\beta}\big)
$$

La métrica de Kerr no depende de $t$ ni de $\phi$, sólo de $r$ y $\theta$.

En el plano ecuatorial y para $\mu = r$, ($\alpha,\beta \in \{t,\phi\}$), se puede ver que (por los ejercicios del cuaderno de DEP):

$$
\begin{aligned}
\Gamma^r_{TT} &= -\frac{1}{2} g^{rr}\partial_r g_{TT}, \\[1mm]
\Gamma^r_{T\phi} &= -\frac{1}{2} g^{rr}\partial_r g_{T\phi}, \\[1mm]
\Gamma^r_{\phi\phi} &= -\frac{1}{2} g^{rr}\partial_r g_{\phi\phi}
\end{aligned}
$$

Sustituyendo esto en (2):

$$
-\frac{1}{2}g^{rr}\left[\partial_r g_{tt} + 2\Omega\partial_r g_{t\phi} + \Omega^2 \partial_r g_{\phi\phi}\right] = 0
$$

Como $g_{rr} \neq 0$, podemos factorizar y simplificar de modo que:

$$
 \partial_r g_{TT} + 2\Omega\partial_r g_{T\phi} + \Omega^2 \partial_r g_{\phi\phi} = 0 \tag{3}
$$



Usemos ahora las componentes de la métrica de Kerr en el plano ecuatorial ($θ=\pi/2$) y usando unidades relativistas:

*Componente $g_{TT} = 1 - \dfrac{2s}{r}$:*

$$
\partial_r g_{TT} = \frac{d}{dr}\left(1 - \frac{2s}{r}\right) = \frac{2s}{r^2}
$$

*Componente $g_{T\phi} = \dfrac{2sa}{r}$:*

$$
\partial_r g_{T\phi} = \frac{d}{dr}\left(\frac{2sa}{r}\right) = -\frac{2sa}{r^2}
$$

*Componente $g_{\phi\phi} = -\left(r^2 + a^2 + \frac{2sa^2}{r}\right)$:*

$$
\partial_r g_{\phi\phi} = -\left(2r - \frac{2sa^2}{r^2}\right) = -2r + \frac{2sa^2}{r^2}
$$

Sustituyendo estas expresiones  en (3):

$$
\frac{2s}{r^2} + 2\Omega\left(-\frac{2sa}{r^2}\right) + \Omega^2\left(-2r + \frac{2sa^2}{r^2}\right) = 0
$$

Dividiendo entre 2:

$$
\frac{s}{r^2} - \frac{2sa}{r^2}\Omega + \left(-r + \frac{sa^2}{r^2}\right)\Omega^2 = 0
$$

Multiplicando todo por $r^2$ y reordenando:

$$
s - 2sa\Omega + \left(-r^3 + sa^2\right)\Omega^2 = 0 \\
(-r^3 + sa^2)\Omega^2 - 2sa\Omega + s = 0
$$

Multiplicando por $(-1)$:

$$
(r^3 - sa^2)\Omega^2 + 2sa\Omega - s = 0 \tag{4}
$$

Esta es la ecuación cuadrática en $Ω$  que determina la frecuencia angular de las órbitas ecuatoriales circulares en Kerr.


**Solución para $Ω_\pm(r_0)$**

Resolvamos entonces esta ecuación para $Ω$.

Definiendo:

$$
A = r^3 - sa^2,\quad B = 2sa,\quad C = -s
$$

Por tanto, la solución general de $A\Omega^2 + B\Omega + C=0$ es:

$$
\Omega = \frac{-B \pm \sqrt{B^2 - 4AC}}{2A}
$$

Calculando el discriminante:

$$
\begin{aligned}
B^2 - 4AC &= (2sa)^2 - 4(r^3 - sa^2)(-s) \\
&= 4s^2a^2 + 4s(r^3 - sa^2) \\
&= 4s^2a^2 + 4sr^3 - 4s^2a^2 \\
&= 4sr^3
\end{aligned}
$$

Por tanto:
$$
\sqrt{B^2 - 4AC} = \sqrt{4sr^3} = 2\sqrt{s}r^{3/2}
$$

Sustituyendo en la fórmula cuadrática:

$$
\Omega = \frac{-2sa \pm 2\sqrt{s}r^{3/2}}{2(r^3 - sa^2)} = \frac{-sa \pm \sqrt{s}r^{3/2}}{r^3 - sa^2}
$$

Factorizando:

$$
r^3 - sa^2 = (r^{3/2} - a\sqrt{s})(r^{3/2} + a\sqrt{s})
$$

*Parte prograda (+)*

Tomamos el signo (+) en el numerador:

$$
\begin{aligned}
\Omega_{+} &= \frac{-sa + \sqrt{s}r^{3/2}}{(r^{3/2} - a\sqrt{s})(r^{3/2} + a\sqrt{s})} \\
&= \frac{\sqrt{s}(r^{3/2} - a\sqrt{s})}{(r^{3/2} - a\sqrt{s})(r^{3/2} + a\sqrt{s})} \\
&= \frac{\sqrt{s}}{r^{3/2} + a\sqrt{s}}
\end{aligned}
$$

Esta es la frecuencia angular de la órbita ecuatorial prógrada.

*Parte retrógrada (-)*

Por otro lado si tomamos el signo negativo obtenemos una órbita ecuatorial retrógada:
$$
\Omega_{-} = -\frac{\sqrt{s}}{r^{3/2} - a\sqrt{s}}
$$


**Normalización de la cuadrivelocidad**

Para una partícula que se mueve en la métrica de Kerr (en coordenadas $(T,r,\theta,\phi)$), la cuadrivelocidad $U^\mu = dx^\mu/d\tau$ debe cumplir la condición de normalización:

$$
g_{\mu\nu} U^\mu U^\nu = 1
$$

En general, cerca del punto inicial $(T_0, r_0, \theta_0, \phi_0)$ podemos escribir las componentes de la cuadrivelocidad en términos de velocidades coordenadas con respecto a $T$:

$$
U^\mu_0 = \big(U^T_0,\;U^r_0,\;U^\theta_0,\;U^\phi_0\big)
$$

Pero por definición, la cuadrivelocidad es:

$$
U^\mu = \frac{dx^\mu}{d\tau} = \left(\frac{dT}{d\tau}, \frac{dr}{d\tau}, \frac{d\theta}{d\tau}, \frac{d\phi}{d\tau}\right)
$$

Además, las velocidades coordenadas respecto a $T$ son:

$$
v_r = \frac{dr}{dT}, \qquad v_\theta = \frac{d\theta}{dT}, \qquad \Omega = \frac{d\phi}{dT}
$$

Aplicando la regla de la cadena:

$$
U^r = \frac{dr}{d\tau} = \frac{dr}{dT}\,\frac{dT}{d\tau} = v_r\,U^T
$$

similarmente:

$$
U^\theta = v_\theta U^T, \qquad U^\phi = \Omega U^T
$$

En el instante inicial escribimos entonces

$$
U^\mu = \big(U^T_0,\;v_r\,U^T_0,\;v_\theta\,U^T_0,\;\Omega\,U^T_0\big),
$$

evaluadas en el instante inicial y sustituyendo en la condición de normalización:

$$
\begin{aligned}
g_{\mu\nu} U^\mu_0 U^\nu_0 &= g_{TT} (U^T_0)^2 + 2 g_{T\phi} U^T_0 U^\phi_0 + g_{rr} (U^r_0)^2 + g_{\theta\theta} (U^\theta_0)^2 + g_{\phi\phi} (U^\phi_0)^2 \\[2mm]
&= (U^T_0)^2 \Big[ g_{TT} + 2 g_{T\phi}\,\Omega + g_{rr}\,v_r^2 + g_{\theta\theta}\,v_\theta^2 + g_{\phi\phi}\,\Omega^2 \Big] = 1
\end{aligned}
$$

Definiendo:

$$
A \equiv g_{TT} + 2 g_{T\phi}\,\Omega + g_{rr}\,v_r^2 + g_{\theta\theta}\,v_\theta^2 + g_{\phi\phi}\,\Omega^2
$$

con los componentes métricos evaluados en $(r_0, \theta_0, \phi_0)$, luego, la normalización será:

$$
(U^T_0)^2\,A = 1
$$

entonces
$$
 U^T_0 = \frac{1}{\sqrt{A}}
$$

Una vez conocido $U^T_0$, las demás componentes iniciales se obtienen como:

$$
U^r_0 = v_r\,U^T_0,\qquad U^\theta_0 = v_\theta\,U^T_0,\qquad U^\phi_0 = \Omega\,U^T_0
$$



**Caso particular: órbita circular ecuatorial**

Usando las condiciones de simetría y órbita circular:

$$
U^r_0 = 0, \quad U^\theta_0 = 0, \quad U^\phi_0 = \Omega\,U^T_0
$$

tenemos:

$$
\begin{aligned}
g_{\mu\nu} U^\mu_0 U^\nu_0
&= g_{TT} (U^T_0)^2 + 2 g_{T\phi} U^T_0 U^\phi_0 + g_{\phi\phi} (U^\phi_0)^2 \\[2mm]
&= g_{TT} (U^T_0)^2 + 2 g_{T\phi} (U^T_0)(\Omega U^T_0) + g_{\phi\phi} (\Omega U^T_0)^2 \\[2mm]
&= (U^T_0)^2 \left[g_{TT} + 2 g_{T\phi}\Omega + g_{\phi\phi}\Omega^2\right] = 1
\end{aligned}
$$

Definiendo ahora $A$ como:

$$
A \equiv g_{TT} + 2 g_{T\phi}\Omega + g_{\phi\phi}\Omega^2,
$$

donde $g_{TT}, g_{T\phi}, g_{\phi\phi}$ son los componentes de la métrica aproximada evaluados en $(r_0, \theta_0=\pi/2, \phi_0=0)$.

Entonces la normalización se reduce a:

$$
(U^T_0)^2 A = 1
$$
Por tanto
$$
 U^T_0 = \frac{1}{\sqrt{A}}
$$

y, en consecuencia, la componente $\phi$ es:

$$
U^\phi_0 = \Omega\,U^T_0 = \frac{\Omega}{\sqrt{A}}
$$

En el caso de una órbita ecuatorial circular se tiene que $v_r = 0$, $v_\theta = 0$ y $v_\phi = Ω$.

En el caso de órbitas no circulares podemos tomar $v_r$ y $v_θ$ diferentes de cero.

**Vector de condiciones iniciales (para una órbita circular)**

Finalmente, el estado inicial completo para la integración numérica de las geodésicas puede escribirse como:

$$Y_0 = \left( T(0),\, r(0),\, \theta(0),\, \phi(0),\ U^T(0),\, U^r(0),\, U^\theta(0),\, U^\phi(0) \right), $$

de este modo, si la órbita  es ecuatorial circular, el vector estará dado por:

 $$ Y_0 = \left( T_0, r_0, \frac{\pi}{2},0,  \frac{1}{\sqrt{A}}, 0, 0, \frac{\Omega}{\sqrt{A}} \right)  $$



In [ ]:
# Condiciones iniciales
T0 = 0.0
theta0 = np.pi / 2  # órbita ecuatorial
phi0 = 0.0

In [ ]:
def condiciones_iniciales_orbita_ecuatorial(r0, s_, a_, vr0=0.0, vtheta0=0.0):
  # Frecuencia angular prograde tipo Kerr
  Omega = np.sqrt(s_) / (r0**1.5 + a_ * np.sqrt(s_))

  # Métrica en el punto inicial
  g0 = g_numerico(T0, r0, theta0, phi0, s_, a_)

  # Componentes de la métrica
  g_tt = g0[0, 0]
  g_tphi = g0[0, 3]
  g_rr = g0[1, 1]
  g_thth = g0[2, 2]
  g_phphi = g0[3, 3]

# Normalización
  A = (g_tt
      + 2.0 * g_tphi * Omega
      + g_rr * vr0**2
      + g_thth * vtheta0**2
      + g_phphi * Omega**2)

  UT0 = 1.0 / np.sqrt(A)
  Ur0 = vr0 * UT0
  Utheta0 = vtheta0 * UT0
  Uphi0 = Omega * UT0

  Y0 = np.array([T0, r0, theta0, phi0, UT0, Ur0, Utheta0, Uphi0])
  return Y0

Trabajaremos en unidades canónicas relativistas, es decir:

$$
G = 1, \quad c=1
$$

donde

$$
G = \frac{U_L^3}{U_T^2 U_M}, \quad
c = \frac{U_L}{U_T}
$$



In [ ]:
c = 1 # pc.constantes.c / UV
G = 1 # pc.constantes.G * UM / UL**3
M = 1 # m / UL

De este modo, el parámetro de Schwarzschild será

$$
s = \frac{GM^2}{c^2} = 1
$$

## Simulaciones de las órbitas

Elegimos un radio inicial

$$r_0 = 4\,s $$es decir, una órbita ecuatorial situada a unos pocos radios de Schwarzschild del cuerpo central, lo suficientemente lejos del horizonte como para que la órbita sea estable en el potencial efectivo.

Como primer caso, tomaremos $a = 0$, de manera que la métrica de Kerr linealizada se reduce a la métrica de Schwarzschild (sin rotación).

 Con estos parámetros:

 $$r_0 = 4, \quad s = 1, \quad a = 0, $$

podemos verificar que el código reproduce una órbita ecuatorial circular en el caso estático, que servirá de referencia para comparar luego con los casos $a \neq 0$.



In [ ]:
# Parámetros físicos
# Inicialmente trabajeremos con una órbita circular (vr=0)
r0 = 4 # Radio inicial
s_ = 1
a_ = 0

v = np.sqrt(s_ / r0)
P = 2.0 * np.pi * r0 / v # Periodo: P = 2π r / v
t_ini = 0.0
t_fin = 3.0 * P # Integramos 3 periodos orbitales
tsol = np.linspace(t_ini, t_fin, 1000)

Realicemos entonces la integración numérica para resolver las ecuaciones de la geodésica en el límite lineal en a.

In [ ]:
Y0 = condiciones_iniciales_orbita_ecuatorial(r0, s_, a_)

# Integramos geodésica usando tsol como parámetro
sol = solve_ivp(
    sistema_geodesicas,
    (t_ini, t_fin),
    Y0,
    t_eval=tsol,
    args=(s_, a_),
    rtol=1e-9,
    atol=1e-9
)

# Resultados de la soluciín de cada componente
Tsol = sol.y[0]
rsol = sol.y[1]
thetasol = sol.y[2]
phisol = sol.y[3]
Ursol = sol.y[4]
Uthetasol = sol.y[5]
Uphisol = sol.y[6]

En la métrica de Kerr usamos coordenadas tipo esféricas $(r, \theta, \phi)$ (Boyer–Lindquist), pero sabemos que la relación con coordenadas cartesianas $(x, y, z)$ es:

$$
\begin{aligned}
x &= r \sin\theta \cos\phi, \\
y &= r \sin\theta \sin\phi, \\
z &= r \cos\theta
\end{aligned}
$$

Pero si la órbita es ecuatorial, entonces

$$
\theta = \frac{\pi}{2}
$$

por tanto
$$ \sin\theta = 1, \quad \cos\theta = 0
$$

luego:

$$
\begin{aligned}
x &=  r\cos\phi, \\
y &=  r\sin\phi, \\
z &= 0
\end{aligned}
$$

O sea, la partícula se mueve en el plano $z=0$.

In [ ]:
x = rsol * np.cos(phisol)
y = rsol * np.sin(phisol)

**Observación: horizonte de eventos**

En la métrica de Kerr,

$$\Delta = r^2 - 2 s r + (a s)^2$$

En el límite lineal en $a$ (es decir, despreciando términos $\mathcal{O}(a^2)$) tomamos:

$$\Delta ≈ r^2 - 2 s r$$

Pero el horizonte de eventos se define como $\Delta = 0$, así que en esta aproximación:

$$r^2 - 2 s r = 0 $$

por tanto

$$ r_h  \approx 2s$$

In [ ]:
plt.figure(figsize=(8, 8))
r_hor_lin = 2.0 * s_ # Horizonte de eventos en la aproximación lineal
psi = np.linspace(0.0, 2.0*np.pi, 400)
x_h = r_hor_lin * np.cos(psi)
y_h = r_hor_lin * np.sin(psi)
plt.fill(x_h, y_h, 'k--', label='Horizonte (aprox. lineal)')
plt.plot(x, y,color = 'crimson', label=f'a = {a_}')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.title('Órbita ecuatorial en la métrica de Kerr linealizada')

Veamos qué sucede si variamos el parámetro de spin $a$ (tomando valores de $a <1$).

In [ ]:
as_ = [0.001, 0.01, 0.1, 0.3]

In [ ]:
colores = ['dodgerblue', 'crimson', 'seagreen', 'rebeccapurple']
fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(16, 4),
                         sharex=True, sharey=True)

for i, a_val in enumerate(as_):
  ax = axes[i]
  color_actual = colores[i]

  # Condiciones iniciales para cada a
  Y0 = condiciones_iniciales_orbita_ecuatorial(r0, s_, a_val)

  # Integramos geodésica
  sol = solve_ivp(
      sistema_geodesicas,
      (t_ini, t_fin),
      Y0,
      t_eval=tsol,
      args=(s_, a_val),
      rtol=1e-9,
      atol=1e-9
  )

  r_sol = sol.y[1]
  phi_sol = sol.y[3]

  x = r_sol * np.cos(phi_sol)
  y = r_sol * np.sin(phi_sol)
  ax.plot(x, y,color = color_actual)

  # Horizonte linealizado
  ax.fill(x_h, y_h, 'k--', label='Horizonte r = 2s')
  ax.set_aspect('equal', 'box')
  ax.grid(True)
  ax.set_title(f'a = {a_val}')
  # Solo en el primer subplot ponemos etiquetas de ejes
  if i == 0:
    ax.set_xlabel('x')
    ax.set_ylabel('y')
  else:
    ax.set_xlabel('x')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper right')
fig.suptitle('Órbitas ecuatoriales en la métrica de Kerr linealizada', y=1.02)
plt.tight_layout()
plt.show()

Podemos ver que las órbitas son inestables porque para ese radio ($r_0 = 4s$) estamos demasiado cerca del agujero negro como para tener órbitas circulares estables con ese momento angular; por eso, las trayectorias reales tienden hacia el horizonte.

Veamos entonces qué sucede si variamos el valor de $r0$.

In [ ]:
r0s = [4.0, 6.0, 8.0, 10.0]   # radios iniciales a comparar
a_ = 0.1 # spin fijo

fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(16, 4), sharex=True, sharey=True)
for i, r0 in enumerate(r0s):
  ax = axes[i]
  color_actual = colores[i]


  v0 = np.sqrt(s_ / r0)
  P0 = 2.0 * np.pi * r0 / v0
  t_ini = 0.0
  t_fin = 3.0 * P0
  t_eval = np.linspace(t_ini, t_fin, 1000)

  # Condiciones iniciales para c<ada r0
  Y0 = condiciones_iniciales_orbita_ecuatorial(r0, s_, a_)

  # Integramos la geodésica
  sol = solve_ivp(
      sistema_geodesicas,
      (t_ini, t_fin),
      Y0,
      t_eval=t_eval,
      args=(s_, a_),
      rtol=1e-9,
      atol=1e-9
  )

  r_sol = sol.y[1]
  phi_sol = sol.y[3]
  x = r_sol * np.cos(phi_sol)
  y = r_sol * np.sin(phi_sol)

  ax.plot(x, y,color = color_actual)
  ax.fill(x_h, y_h, color='k', label='Horizonte r = 2s')
  ax.set_aspect('equal', 'box')
  ax.grid(True)
  ax.set_title(fr'$r_0 = {r0}$')
  if i == 0:
    ax.set_xlabel('x')
    ax.set_ylabel('y')
  else:
    ax.set_xlabel('x')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels)
fig.suptitle('Órbitas ecuatoriales en la métrica de Kerr linealizada\npara distintos $r_0$ (a fijo)', y=1)
plt.tight_layout()
plt.show()


Podemos ver cómo evoluciona la partícula de prueba orbitando en el plano ecuatorial de un agujero negro de Kerr (en el régimen de rotación lenta), de modo que, al variar el radio inicial $r_0$, se observa una transición en el comportamiento dinámico, mostrando que las órbitas circulares solo son estables a partir de una cierta distancia del centro.

**Condiciones iniciales para una órbita con velocidad radial diferente de cero (órbita no circular)**

Veamos qué sucede si la partícula tiene una velocidad radial inicial distinta de cero, $v_r(0) = dr/dT \neq 0$, es decir que, además de girar alrededor del agujero negro, la partícula puede empezar ya sea cayendo hacia adentro ($v_r(0) < 0$) o alejándose del agujero ($v_r(0) > 0$), por tanot, la órbita deja de ser  circular y se vuelve más general.

Analicemos esto para diferentes valores de $v_r(0)$ para un mismo radio inicial $r_0$ y un mismo parámetro de spin $a$.

In [ ]:
# Parámetros
r0 = 8.0  # radio inicial
aval = 0.1 # spin

# Definimos el tiempo de integración a partir del periodo circular
v_circ = np.sqrt(s_ / r0)
P = 2.0 * np.pi * r0 / v_circ # periodo orbital
t_ini = 0.0
t_fin = 3.0 * P # integramos  durante 3 periodos
tsol = np.linspace(t_ini, t_fin, 1500)

In [ ]:
vr0_vals = [0.0, 0.05, 0.1, 0.15]  # Valores de velocidad radial inicial v_r(0) = dr/dT

fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(16, 4),sharex=True, sharey=True)

for i, vr0 in enumerate(vr0_vals):
  ax = axes[i]
  color_actual = colores[i]

  # Condiciones iniciales para cada vr0
  Y0 = condiciones_iniciales_orbita_ecuatorial(r0, s_, aval,vr0=vr0,vtheta0=0.0)

  # Integramos las geodésicas
  sol = solve_ivp(
      sistema_geodesicas,
      (t_ini, t_fin),
      Y0,
      t_eval=tsol,
      args=(s_, aval),
      rtol=1e-9,
      atol=1e-9
  )
  r_sol = sol.y[1]
  phi_sol = sol.y[3]

  # Pasamos a coordenadas cartesianas en el plano ecuatorial
  x = r_sol * np.cos(phi_sol)
  y = r_sol * np.sin(phi_sol)

  ax.plot(x, y, color = color_actual)
  ax.fill(x_h, y_h, color='k', alpha=0.2) # Horizonte
  ax.set_aspect('equal', 'box')
  ax.grid(True)
  ax.set_title(rf'$v_r(0) = {vr0}$')
  if i == 0:
    ax.set_xlabel('x')
    ax.set_ylabel('y')
  else:
    ax.set_xlabel('x')

fig.suptitle(r'Órbitas ecuatoriales para distintos'+ fr'($r_0={r0}$, $a={aval}$)',y=1.03)
plt.tight_layout()
plt.show()

Vemos que al aumentar $v_r(0)$ las órbitas dejan de ser estables y se vuelven más excéntricas , hasta que llega un punto en el que la componente radial es tan grande que la partícula casi escapa del sistema.

## Verificación constante de movimiento

**Condición de normalización**

Dado que el tiempo propiio $\tau$ es un invariante relativista, la cuadrivelocidad debe satisfacer la restricción algebraica en todo punto de la trayectoria:

$$
g_{\mu\nu}\frac{dx^\mu}{d\tau}\frac{dx^\nu}{d\tau} = 1
$$


In [ ]:
# Calculamos gUU a lo largo de una geodésica
def norma_4velocidad(sol, s_, a_):
  # Extraemos las coordenadas espaciales y de velocidad de la solucion
  T_, r_, th_, phi_ = sol.y[0:4]
  UT_, UR_, Utheta_, Uphi_ = sol.y[4:8]
  n = len(T_)

  # Guardamos gUU en cada punto
  vals = np.zeros(n)
  # Recorremos todos los puntos de la geodésica
  for i in range(n):
    # Métrica evaluada en (T,r,θ,φ)_i
    g = g_numerico(T_[i], r_[i], th_[i], phi_[i], s_, a_)

    # Vector 4-velocidad u^μ en ese punto
    u_vec = np.array([UT_[i], UR_[i], Utheta_[i], Uphi_[i]])

    # Norma: g_{μν}u^μu^ν = u^T g u
    vals[i] = float(u_vec @ (g @ u_vec))
  return vals

Veamos entonces que se cumpla la condición de normalización para cierto valor de $a$

In [ ]:
a_norm = 0.1
r0_norm = r0

# Tiempo de integraión
v_circ = np.sqrt(s_ / r0_norm)
P = 2.0 * np.pi * r0_norm / v_circ
tau_ini = 0.0
tau_fin = 3.0 * P
tau_eval_norm = np.linspace(tau_ini, tau_fin, 2000)

# Condiciones iniciales para una órbita ecuatorial casi circular
Y0_norm = condiciones_iniciales_orbita_ecuatorial(r0_norm, s_, a_norm, vr0=0.0, vtheta0=0.0)

# Integramos la geodésica
sol_norm = solve_ivp(
    sistema_geodesicas,
    (tau_ini, tau_fin),
    Y0_norm,
    t_eval=tau_eval_norm,
    args=(s_, a_norm),
    rtol=1e-9,
    atol=1e-9
)

In [ ]:
# Calculamos la norma gUU a lo largo de la solución
norm_u = norma_4velocidad(sol_norm, s_, a_norm)
norm_u

In [ ]:
plt.figure()
plt.plot(tau_eval_norm, norm_u, color = 'dodgerblue')
plt.axhline(1.0, linestyle='--')
plt.xlabel(r'$\tau$')
plt.ylabel(r'$g_{\mu\nu}U\mu U^\nu$')
plt.title(f'Constancia de la norma de la cuadrivelocidad (a = {a_norm})')
plt.grid(True)
plt.tight_layout()
plt.show()

Podemos ver que la norma de la cuadrivelocidad se mantiene constante alrededor de $1$ a lo largo de toda la trayectoria calculada, además, podemos ver que las variaciones respecto a este valor son del orden de la tolerancia del integrador ($10^{-9}$), por lo cual las variaciones que se ven en la gráfica son  numéricas.

# **4). Métrica de Kerr**

Partimos de la métrica de Kerr en coordenadas de Boyer–Lindquist $(t, r, \theta, \phi)$, dada por:

$$ds^2 = \left(1-\frac{2 s r}{\Sigma}\right) c^2 dt^2 + \frac{4 a c s^2 r \sin ^2 \theta}{\Sigma}\, dt\, d \phi - \frac{\Sigma}{\Delta}\, dr^2 - \Sigma\, d \theta^2 - \frac{\sin ^2 \theta}{\Sigma}\left[\left(r^2+(a s)^2\right)^2-(a s)^2 \Delta \sin ^2 \theta\right] d \phi^2,
$$

donde las funciones auxiliares $\Sigma$ y $\Delta$ se definen como:

$$\Sigma = r^2 + (a s)^2 \cos^2\theta, \qquad \Delta = r^2 - 2 s r + (a s)^2.
$$

El elemento de línea se puede escribir en forma compacta como $ds^2 = g_{\mu\nu}\,dx^\mu dx^\nu$, de modo que los coeficientes $g_{\mu\nu}$ son las componentes del tensor métrico en la base de coordenadas $(t, r, \theta, \phi)$.

Notemos que, debido a la simetría $g_{\mu\nu} = g_{\nu\mu}$, el término cruzado $dt\,d\phi$ proviene de $2 g_{t\phi}\, dt\, d\phi$, por lo que el coeficiente que multiplica $dt\,d\phi$ en $ds^2$ es $2 g_{t\phi}$. En consecuencia, $g_{t\phi}$ es la mitad de dicho coeficiente.

Por tanto, el tensor métrico $g_{\mu\nu}$ se puede escribir matricialmente como:


$$g_{\mu\nu} = \begin{pmatrix}
\left(1-\dfrac{2 s r}{\Sigma}\right) c^2 & 0 & 0 & \dfrac{2 a c s^2 r \sin ^2 \theta}{\Sigma} \\[15pt]
0 & -\dfrac{\Sigma}{\Delta} & 0 & 0 \\[15pt]
0 & 0 & -\Sigma & 0 \\[15pt]
\dfrac{2 a c s^2 r \sin ^2 \theta}{\Sigma} & 0 & 0 & -\dfrac{\sin ^2 \theta}{\Sigma}\left[\left(r^2+(a s)^2\right)^2-(a s)^2 \Delta \sin ^2 \theta\right]
\end{pmatrix}$$

Pero por simplicidad trabajaremos con $T=ct$, de este modo,el invariante $ds^2$ en las coordenadas $(T, r, \theta, \phi)$ queda:

$$ds^2 = \left(1-\frac{2 s r}{\Sigma}\right) dT^2 + \frac{4 a s^2 r \sin ^2 \theta}{\Sigma} dT d\phi - \frac{\Sigma}{\Delta} dr^2 - \Sigma d \theta^2 - \frac{\sin ^2 \theta}{\Sigma}\left[\left(r^2+(a s)^2\right)^2-(a s)^2 \Delta \sin ^2 \theta\right] d \phi^2$$


Matricialmente:

$$g_{\mu\nu} = \begin{pmatrix}
1-\dfrac{2 s r}{\Sigma} & 0 & 0 & \dfrac{2 a s^2 r \sin ^2 \theta}{\Sigma} \\[15pt]
0 & -\dfrac{\Sigma}{\Delta} & 0 & 0 \\[15pt]
0 & 0 & -\Sigma & 0 \\[15pt]
\dfrac{2 a s^2 r \sin ^2 \theta}{\Sigma} & 0 & 0 & -\dfrac{\sin ^2 \theta}{\Sigma}\left[\left(r^2+(a s)^2\right)^2-(a s)^2 \Delta \sin ^2 \theta\right]
\end{pmatrix}$$


De este modo, las componentes de la métrica de Kerr serán:

**Componente temporal ($g_{TT}$):**
$$g_{TT} = 1-\frac{2 s r}{\Sigma}$$

**Componente radial ($g_{rr}$):**
$$g_{rr} = -\frac{\Sigma}{\Delta}$$

**Componentes angulares ($g_{\theta\theta}$ y $g_{\phi\phi}$):**
$$g_{\theta\theta} = -\Sigma$$$$g_{\phi\phi} = -\frac{\sin ^2 \theta}{\Sigma}\left[\left(r^2+(a s)^2\right)^2-(a s)^2 \Delta \sin ^2 \theta\right]$$

**Término cruzado ($g_{T\phi}$):**
$$g_{T\phi} = \frac{2 a s^2 r \sin ^2 \theta}{\Sigma}$$

Veamos esto simbólicamente

In [ ]:
T, r, theta, phi, a, s = sp.symbols('T r theta phi a s')
Sigma = r**2 + (a * s)**2 * sp.cos(theta)**2
Delta = r**2 - 2 * s * r + (a * s)**2

In [ ]:
# Componentes de la métrica
# Componente temporal
g_TT = 1 - 2 * s * r / Sigma

# Componente radial
g_rr = -Sigma / Delta

# Componentes angulares
g_ThetaTheta = -Sigma
g_PhiPhi = - sp.sin(theta)**2 / Sigma * (
    (r**2 + (a * s)**2)**2
    - (a * s)**2 * Delta * sp.sin(theta)**2)

# Término cruzado
g_Tphi = 2 * a * s**2 * r * sp.sin(theta)**2 / Sigma

In [ ]:
# Matriz de la métrica de kerr
g_kerr = [
    [g_TT, 0, 0, g_Tphi],
    [0, g_rr, 0, 0],
    [0, 0, g_ThetaTheta, 0],
    [g_Tphi, 0, 0, g_PhiPhi]
]

g_kerr_metrica = es.MetricTensor(g_kerr, (T, r, theta, phi))
g_kerr_metrica.tensor()

Convirtamos entonces la métrica simbólica a una métrica numérica

In [ ]:
vars_gkerr, g_kerr_func = g_kerr_metrica.tensor_lambdify() # Métrica numérica

def gkerr_numerico(Tk, rk, thetak, phik, ak, sk):
  subs = {T: Tk, r: rk, theta: thetak, phi: phik, a: ak, s: sk}
  args = []
  for var in vars_gkerr:
    valor = subs[var]
    args.append(valor)
  return np.array(g_kerr_func(*args))

Los símbolos de Christoffel para la métrica de Kerr serán:

In [ ]:
# Símbolos de Chris para la métrica de Kerr
Gamma_kerr = es.ChristoffelSymbols.from_metric(g_kerr_metrica)
Gamma_kerr_tensor = Gamma_kerr.tensor()
Gamma_kerr_tensor

Veamos si podemos simplificarlos

In [ ]:
Gamma_kerr_tensor = Gamma_kerr_tensor.applyfunc(sp.simplify)
Gamma_kerr_tensor

Como se vio en el punto anterior, para poder integrar numéricamente, necesitamos pasar las funciones simbólicas a funciones numéricas:

In [ ]:
Gammakerr_func = []
Gammakerr_vars = []

for mu in range (4):
  Gammakerr_matrix = sp.Matrix(Gamma_kerr_tensor[mu, :, :])
  aux_metric = es.MetricTensor(Gammakerr_matrix.tolist(), (T, r, theta, phi))
  vars_Gammakerr, gammakerr_func = aux_metric.tensor_lambdify()
  Gammakerr_func.append(gammakerr_func)
  Gammakerr_vars.append(vars_Gammakerr)

In [ ]:
def Gammakerr_numerico(Tk, rk, thetak, phik, ak, sk):
  subs = {T: Tk, r: rk, theta: thetak, phi: phik, a: ak, s: sk}
  Gammakerr_num = np.zeros((4, 4, 4)) # Tensor numérico
  for mu in range(4):
    args = []
    for var in Gammakerr_vars[mu]:
      valor = subs[var]
      args.append(valor)
    Gammakerr_num[mu, :, :] = np.array(Gammakerr_func[mu](*args))
  return Gammakerr_num

Con esto podemos definir el sistema de ecuaciones diferenciales corrrespondiente a la métrica de Kerr

In [ ]:
def sistema_geodesicas_kerr(lamda, Y, ak, sk):
  # Coordenadas y 4-velocidad
  Tk, rk, thetak, phik, UT, UR, Utheta, Uphi = Y
  U = np.array([UT, UR, Utheta, Uphi])
  Gamma = Gammakerr_numerico(Tk, rk, thetak, phik, ak, sk)

  dU = np.zeros(4)
  for mu in range(4):
    suma = 0.0
    for alpha in range(4):
      for beta in range(4):
        suma += Gamma[mu, alpha, beta] * U[alpha] * U[beta]
    dU[mu] = -suma

  dYdtau = np.zeros(8)
  dYdtau[:4] = U
  dYdtau[4:] = dU
  return dYdtau

In [ ]:
T0 = 0.0
def condiciones_iniciales_generales(r0, theta0, phi0, ak, sk, vr0=0.0, vtheta0=0.0, Omega=0.0):
  g0 = gkerr_numerico(T0, r0, theta0, phi0, ak, sk)
  g_tt = g0[0,0]
  g_tphi = g0[0,3]
  g_rr = g0[1,1]
  g_thth = g0[2,2]
  g_phph = g0[3,3]

  A = (g_tt
        + 2.0 * g_tphi * Omega
        + g_rr * vr0**2
        + g_thth * vtheta0**2
        + g_phph * Omega**2)

  UT0 = 1.0 / np.sqrt(A) # componente temporal de la cuadrivelocidad
  Ur0 = vr0 * UT0
  Utheta0 = vtheta0 * UT0
  Uphi0 = Omega   * UT0

  Y0 = np.array([T0, r0, theta0, phi0, UT0, Ur0, Utheta0, Uphi0])
  return Y0

Alicemos el comportamiento y la trayectoria que sigue la partícula para diferentes órbitas

## **Órbita circular**

Como vimos en el punto anterior, para órbitas circulares ecuatoriales de Kerr, la frecuencia angular está dada por:

$$\Omega_\pm(r_0) = \frac{\sqrt{s}}{r_0^{3/2} \pm a\sqrt{s}}$$

De modo que el signo indica el sentido de la órbita (prógado, retrógado)

In [ ]:
def Omega_circular_kerr(r0, ak, sk, sentido="+"):

  # Órbita prógrada
  if sentido == "+":
    den = r0**(3/2) + ak * np.sqrt(sk)
    num = np.sqrt(sk)
  # órbita retrógada
  else:
    num = np.sqrt(sk)
    den = r0**(3/2) - ak * np.sqrt(sk)

  Omega = num / den
  return Omega

Recordemos que para órbitas ecuatoriales necesariamente $\theta_0 = \pi/2$.

En este caso estaos asumiendo una órbita circular ideal, de modo que $v_r = v_\theta = 0$.

In [ ]:
# Definimos las condiciones inicales explicada en el punto 3 para una órbita ecuatorial (prógada o retrógada)

def ecuatorial_prograda(r0, ak, sk):
  theta0 = np.pi/2
  phi0 = 0.0
  Omega = Omega_circular_kerr(r0, ak, sk, sentido="+")
  return condiciones_iniciales_generales(r0, theta0, phi0,
                                          ak, sk,
                                          vr0=0.0, vtheta0=0.0,
                                          Omega=Omega)

def ecuatorial_retrograda(r0, ak, sk):
  theta0 = np.pi/2
  phi0 = 0.0
  Omega = Omega_circular_kerr(r0, ak, sk, sentido="-")
  return condiciones_iniciales_generales(r0, theta0, phi0,
                                          ak, sk,
                                          vr0=0.0, vtheta0=0.0,
                                          Omega=Omega)

  Veamos entonces el comportamiento de las órbitas para cada uno de los casos

### Órbita ecuatorial prógrada

In [ ]:
sk = 1
ak = 0.7
r0 = 8

# Definimos la escala de tiempo a partir del pe´riodo
v_orb = np.sqrt(sk / r0)
P = 2.0 * np.pi * r0 / v_orb
tau_i = 0
tau_f = 4* P
tau_eval = np.linspace(tau_i, tau_f, 3000)

In [ ]:
Y0_prog = ecuatorial_prograda(r0, ak, sk)
sol_prog = solve_ivp(
    sistema_geodesicas_kerr,
    (tau_i, tau_f),
    Y0_prog,
    t_eval = tau_eval,
    args = (ak, sk),
    rtol = 1e-9,
    atol = 1e-9
)

r_prog = sol_prog.y[1]
theta_prog = sol_prog.y[2]
phi_prog = sol_prog.y[3]

In [ ]:
# Función para pasar de Boyer-Lindquist a cartesianas
def bl2cart(r_, theta_, phi_):
  x = r_ * np.sin(theta_) * np.cos(phi_)
  y = r_ * np.sin(theta_) * np.sin(phi_)
  z = r_ * np.cos(theta_)
  return x, y, z

In [ ]:
x_prog, y_prog, z_prog = bl2cart(r_prog, theta_prog, phi_prog)

#### **Observación: Horizonte de eventos en la métrica de Kerr**

Sabemos que los horizontes de eventos de la solución de Kerr se localizan en los radios donde $\Delta(r)=0$, entonces:

$$
\Delta(r) = r^2 - 2 s r + (a s)^2 = 0
$$

Esto es una ecuación cuadrática en $r$ del tipo $Ar^2 + Br + C = 0$, donde:

$$
A = 1, \qquad B = -2s, \qquad C = (a s)^2
$$

Aplicando la fórmula general y sustituyendo valores e tiene que:

$$
\begin{aligned}
r &= \frac{2s \pm \sqrt{4s^2 - 4 (a s)^2}}{2} \\[2mm]
&= \frac{2s \pm 2\sqrt{s^2 - (a s)^2}}{2} \\[2mm]
&= s \pm \sqrt{s^2 - (a s)^2}
\end{aligned}
$$

Con esto definimos entonces dos radios de singularidaes:
$$
r_\pm = s \pm \sqrt{s^2 - (a s)^2}
$$

Factorizando y tomando solo la parte positiva se tiene:

$$
r_+ = s\left(1 + \sqrt{1 - a^2}\right)
$$


In [ ]:
r_h = sk * (1 + np.sqrt(1 - ak**2))
psi_kerr = np.linspace(0, 2 * np.pi, 400)
xh_kerr = r_h * np.cos(psi_kerr)
yh_kerr = r_h * np.sin(psi_kerr)

In [ ]:
# Órbita ecuatorial prógrada
plt.figure(figsize=(8, 8))
plt.fill(xh_kerr, yh_kerr, color='k', label='Horizonte de eventos') # Horizonte
plt.plot(x_prog, y_prog,color = 'crimson', label='Ecuatorial prógrada')
plt.xlabel('x')
plt.ylabel('y')
plt.gca().set_aspect('equal', 'box')
plt.grid(True)
plt.legend()
plt.title('Órbita ecuatorial prógrada')
plt.tight_layout()
plt.show()

Veamos qué sucede para diferentes valores de $r_0$

In [ ]:
ak = 0.2
r0s = [3.0, 5.0, 7.0, 9.0]

plt.figure(figsize=(8, 8))
plt.fill(x_h, y_h, alpha=0.3, color='k', label='Horizonte de eventos')

for r0 in r0s:
  # Frecuencia y periodo prógrado
  Omega_pro = Omega_circular_kerr(r0, ak, sk, sentido="+")
  P_pro = 2.0 * np.pi / Omega_pro
  tau_ini = 0.0
  tau_fin = 3.0 * P_pro
  tau_eval = np.linspace(tau_ini, tau_fin, 3000)
  # Condiciones iniciales prógradas
  Y0 = ecuatorial_prograda(r0, ak, sk)

  # Integramos la geodésica
  sol = solve_ivp(
      sistema_geodesicas_kerr,
      (tau_ini, tau_fin),
      Y0,
      t_eval=tau_eval,
      args=(ak, sk),
      rtol=1e-9,
      atol=1e-9
  )

  r_arr, th_arr, phi_arr = sol.y[1], sol.y[2], sol.y[3]
  x, y, z = bl2cart(r_arr, th_arr, phi_arr)
  line, = plt.plot(x, y, label=fr'r0 = {r0}')
  col = line.get_color()
  # Punto de inicio (τ = 0)
  plt.scatter(x[0],  y[0],  color=col, marker='o', s=40, zorder=5)
  # Punto de fin (τ = τ_final)
  plt.scatter(x[-1], y[-1], color=col, marker='x', s=40, zorder=5)

plt.xlabel('x')
plt.ylabel('y')
plt.gca().set_aspect('equal', 'box')
plt.grid(True)
plt.legend()
plt.title(f'Órbitas ecuatoriales prógradas (a = {ak})')
plt.tight_layout()
plt.show()

Los puntos indican la posición inicial de cada partícula y las cruces la posición final. Para las órbitas con $r_0 = 5, 7, 9$ las trayectorias son casi círculos perfectos y los puntos de inicio y fin prácticamente coinciden, lo que indica órbitas muy estables y casi perfectamente periódicas, pero también podemos ver que cuanto más cerca está la órbita del agujero negro (menor $r_0$), mayor es la frecuencia angular, y que si el radio está muy próximo al horizonte (como el caso $r_0 = 3$) la trayectoria deja de ser cerrada y la partícula termina siendo absorbida por el agujero negro.

In [ ]:
# El siguiente código fue generado con ayuda del asistente de IA de Google
# Prompt: "Realiza un gráfico usando plotly para diferentes valores
#de r0s de una partícula en una órbita prógrada"
import plotly.graph_objects as go

# Parámetros de la métrica
ak = 0.2
sk = 1.0
r0s = [3.0, 5.0, 7.0, 9.0]

# Horizonte de eventos de Kerr (r_+) como esfera 3D
r_plus = sk + np.sqrt(sk**2 - (ak * sk)**2)

u = np.linspace(0, 2*np.pi, 60)
v = np.linspace(0, np.pi, 30)

xh = r_plus * np.outer(np.sin(v), np.cos(u))
yh = r_plus * np.outer(np.sin(v), np.sin(u))
zh = r_plus * np.outer(np.cos(v), np.ones_like(u))

fig = go.Figure()
# Superficie del horizonte
fig.add_trace(
    go.Surface(
        x=xh,
        y=yh,
        z=zh,
        colorscale='Greys',
        showscale=False,
        name='Horizonte de eventos'
    )
)

# Órbitas ecuatoriales prógradas para distintos r0
for r0 in r0s:
    # Frecuencia y periodo prógrado
    Omega_pro = Omega_circular_kerr(r0, ak, sk, sentido="+")
    P_pro = 2.0 * np.pi / Omega_pro

    tau_ini = 0.0
    tau_fin = 3.0 * P_pro   # puedes subir a 5–8 P si quieres trayectorias más largas
    tau_eval = np.linspace(tau_ini, tau_fin, 4000)

    # Condiciones iniciales prógradas
    Y0 = ecuatorial_prograda(r0, ak, sk)

    # Integramos la geodésica
    sol = solve_ivp(
        sistema_geodesicas_kerr,
        (tau_ini, tau_fin),
        Y0,
        t_eval=tau_eval,
        args=(ak, sk),
        rtol=1e-9,
        atol=1e-9
    )

    r_arr, th_arr, phi_arr = sol.y[1], sol.y[2], sol.y[3]
    x, y, z = bl2cart(r_arr, th_arr, phi_arr)

    # Curva 3D de la órbita
    fig.add_trace(
        go.Scatter3d(
            x=x,
            y=y,
            z=z,
            mode='lines',
            name=fr'r0 = {r0}'
        )
    )

    # Punto inicial
    fig.add_trace(
        go.Scatter3d(
            x=[x[0]],
            y=[y[0]],
            z=[z[0]],
            mode='markers',
            marker=dict(symbol='circle', size=4),
            showlegend=False
        )
    )

# Layout 3D
fig.update_layout(
    title=f'Órbitas ecuatoriales prógradas en 3D (a = {ak})',
    scene=dict(
        xaxis_title='x',
        yaxis_title='y',
        zaxis_title='z',
        aspectmode='data'   # misma escala en x,y,z
    ),
    width=800,
    height=700
)

fig.show()

### Órbitas ecuatoriales retrógadas

Realicemos ahora un procedimiento similar al anterior pero para órbitas retrógadas, es decir cuando:

$$Ω = \frac{\sqrt{s}}{r_0^{3/2} + a \sqrt{s}}$$

In [ ]:
ak = 0.2
r0s = [5.0, 7.0, 9.0]

plt.figure(figsize=(8, 8))
plt.fill(x_h, y_h, color='k', label='Horizonte de eventos')

for r0 in r0s:
  Omega_ret = Omega_circular_kerr(r0, ak, sk, sentido="-")
  P_ret = 2.0 * np.pi / Omega_ret
  tau_ini = 0.0
  tau_fin = 3.0 * P_ret
  tau_eval = np.linspace(tau_ini, tau_fin, 3000)
  # Condiciones iniciales retrógradas para cada r0
  Y0 = ecuatorial_retrograda(r0, ak, sk)
  # Integramos la geodésica en la métrica de Kerr completa
  sol = solve_ivp(
      sistema_geodesicas_kerr,
      (tau_ini, tau_fin),
      Y0,
      t_eval=tau_eval,
      args=(ak, sk),
      rtol=1e-9,
      atol=1e-9
  )
  r_arr, th_arr, phi_arr = sol.y[1], sol.y[2], sol.y[3]
  x, y, z = bl2cart(r_arr, th_arr, phi_arr)
  plt.plot(x, y, label=fr' r0 = {r0}')

plt.xlabel('x')
plt.ylabel('y')
plt.axis('equal')
plt.grid(True)
plt.legend()
plt.title(f'Órbitas ecuatoriales retrógradas (a = {ak})')
plt.tight_layout()
plt.show()

Podemos ver que ya las órbitas no son círculos perfectos sino órbitas abiertas, esto puede deberse a que ahora la partícula va en sentido contrario a la rotación del cuerpo central, es decir, el movimiento va en contra del arrastre del espacio tiempo.

## **Órbita polar**

Veamos ahora qué sucede para aquelals órbitas en las que el plano orbital contiene al eje de rotación del agujero negro, donde la partícula no se mueve solo en el ecuador sino también en los polos.

Para este caso se tiene que:

$$
v_r(0) = 0, \qquad v_\theta(0) \neq 0,\qquad \Omega(0) \approx 0
$$

In [ ]:
def polar(r0, theta0, ak, sk, vr0=0.0, vtheta0=0.0, Omega0=0.02, phi0=0):
  return condiciones_iniciales_generales(
        r0, theta0, phi0,
        ak, sk,
        vr0=vr0,
        vtheta0=vtheta0,
        Omega=Omega0
    )

In [ ]:
ak = 0.5
r0 = 10.0 # radio inicial de la órbita polar

# Condiciones iniciales polares
theta0_pol = 0.3  # Cercano al eje
vtheta0_pol = 0.02  # Pequeña velocidad en theta
Omega0_pol = 0.03  # Casi sin momento angular azimutal
vr0_pol = 0.0
phi0_pol = 0.0

# Horzonte de eventos
r_pol = sk + np.sqrt(sk**2 - (ak * sk)**2)
u = np.linspace(0, 2*np.pi, 40)
v = np.linspace(0, np.pi, 20)
xh = r_pol * np.outer(np.sin(v), np.cos(u))
yh = r_pol * np.outer(np.sin(v), np.sin(u))
zh = r_pol * np.outer(np.cos(v), np.ones_like(u))

In [ ]:
P_pol = 2.0 * np.pi / Omega0_pol
tau_ini = 0.0
tau_fin = 5.0 * P_pol
tau_eval = np.linspace(tau_ini, tau_fin, 8000)

# Evaluamos en las condiciones iniciales polares
Y0 = polar(
    r0, theta0_pol,
    ak, sk,
    vr0=vr0_pol,
    vtheta0=vtheta0_pol,
    Omega0=Omega0_pol,
    phi0=phi0_pol
)

sol = solve_ivp(
    sistema_geodesicas_kerr,
    (tau_ini, tau_fin),
    Y0,
    t_eval=tau_eval,
    args=(ak, sk),
    rtol=1e-9,
    atol=1e-9
)

r_arr, th_arr, phi_arr = sol.y[1], sol.y[2], sol.y[3]
x, y, z = bl2cart(r_arr, th_arr, phi_arr)

In [ ]:
fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(xh, yh, zh, color='black',  alpha = 0.4)
ax.plot(x, y, z, color='crimson', label=fr'Órbita polar, r0 = {r0}')
ax.scatter(x[0], y[0], z[0], color='black', marker='o', s=40, label='inicio')
ax.scatter(x[-1], y[-1], z[-1], color='dodgerblue', marker='x', s=60, label='fin')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('z')
ax.set_title(f'Órbita polar (a = {ak})')
ax.legend()
plt.tight_layout()
plt.show()

Veamos qué sucede para varios valores $r_0$:

In [ ]:
# Código realizado con ayuda de la IA de google
ak = 0.7
r0s = [3, 5, 7, 15]

# Condiciones "polares"
theta0_pol  = 0.3
vtheta0_pol = 0.02
Omega0_pol  = 0.0
vr0_pol     = 0.0
phi0_pol    = 0.0

# Horizonte de eventos de Kerr (r_+) como esfera 3D
r_plus = sk + np.sqrt(sk**2 - (ak * sk)**2)

u = np.linspace(0, 2*np.pi, 60)
v = np.linspace(0, np.pi, 30)

xh = r_plus * np.outer(np.sin(v), np.cos(u))
yh = r_plus * np.outer(np.sin(v), np.sin(u))
zh = r_plus * np.outer(np.cos(v), np.ones_like(u))

fig = go.Figure()

# Superficie del horizonte
fig.add_trace(
    go.Surface(
        x=xh,
        y=yh,
        z=zh,
        colorscale='Greys',
        showscale=False,
        opacity=0.3,
        name='Horizonte de eventos'
    )
)

# ÓRBITAS POLARES PARA DISTINTOS r0
for r0 in r0s:
    # Escala de tiempo: aprox newtoniana como referencia
    Omega_newt = np.sqrt(sk / r0**3)
    P_newt = 2.0 * np.pi / Omega_newt

    tau_ini = 0.0
    tau_fin = 3.0 * P_newt   # puedes subir a 8–10 P_newt para órbitas más largas
    tau_eval = np.linspace(tau_ini, tau_fin, 8000)

    # Condiciones iniciales polares para este r0
    Y0 = polar(
        r0, theta0_pol,
        ak, sk,
        vr0=vr0_pol,
        vtheta0=vtheta0_pol,
        Omega0=Omega0_pol,
        phi0=phi0_pol
    )

    # Integramos la geodésica
    sol = solve_ivp(
        sistema_geodesicas_kerr,
        (tau_ini, tau_fin),
        Y0,
        t_eval=tau_eval,
        args=(ak, sk),
        rtol=1e-9,
        atol=1e-9
    )

    r_arr, th_arr, phi_arr = sol.y[1], sol.y[2], sol.y[3]
    x, y, z = bl2cart(r_arr, th_arr, phi_arr)

    # Curva 3D de la órbita polar
    fig.add_trace(
        go.Scatter3d(
            x=x,
            y=y,
            z=z,
            mode='lines',
            name=fr'polar, r0 = {r0}'
        )
    )

    # Punto inicial
    fig.add_trace(
        go.Scatter3d(
            x=[x[0]],
            y=[y[0]],
            z=[z[0]],
            mode='markers',
            marker=dict(symbol='circle', size=4),
            showlegend=False
        )
    )

# Layout 3D
fig.update_layout(
    title=f'Órbitas polares en 3D (a = {ak})',
    scene=dict(
        xaxis_title='x',
        yaxis_title='y',
        zaxis_title='z',
        aspectmode='data'   # misma escala en x, y, z
    ),
    width=800,
    height=700
)

fig.show()

## Declaración de uso IAs.

*Declaro que la mayor parte del código contenido en este notebook es de mi propia autoría y no ha sido obtenido usando exclusivamente grandes modelos de lenguaje (LLMs o IAs)*.

*En casos excepcionales*:

- *Las celdas con gráficos complejos en `plotly` han sido elaboradas con el apoyo del modelo Gemini que viene incorporado en `Google Colab`*.
- *Cualquier uso adicional fue declarado en las celdas respectivas y en caso de que grandes porciones de código hayan sido extraídas de una IA, lo he indicado explícitamente agregando entre los comentarios el o los _prompts_ utilizados.*